# CICIDS2017 stratified sampling

Creates a small local-testing sample of `datasets/cicids2017_cleaned.csv`.

Strategy:
1. Sample `SAMPLE_FRACTION` of each label.
2. Keep at least `MIN_PER_CLASS` rows per label, or all rows when fewer are available. This increases the share of rare labels relative to the original dataset.
3. Shuffle the sample with a fixed seed.

## Configure

Run this notebook from the `notebooks/` directory. Run the cleaning notebook first if the cleaned CSV and label mapping are unavailable.

In [ ]:
import json
from pathlib import Path

import pandas as pd

DATASETS_DIR = Path("../datasets")
INPUT_PATH = DATASETS_DIR / "cicids2017_cleaned.csv"
OUTPUT_PATH = DATASETS_DIR / "cicids2017_10perc_sample.csv"
LABEL_MAPPING_PATH = DATASETS_DIR / "cicids2017_label_mapping.json"

LABEL_COLUMN = "Label"
SAMPLE_FRACTION = 0.1
MIN_PER_CLASS = 500
SEED = 42

## Load data and inspect labels

Load the full cleaned dataset and display the original number of rows per label.

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f"Rows: {len(df):,}  Columns: {df.shape[1]}")

label_names = {v: k for k, v in json.loads(LABEL_MAPPING_PATH.read_text()).items()}
original_counts = df[LABEL_COLUMN].value_counts().sort_index()
original_counts.rename(index=label_names)

## Sample each label

Apply the configured fraction and per-class minimum, then shuffle the selected rows.

In [ ]:
def stratified_sample(
    data: pd.DataFrame,
    label_column: str,
    fraction: float,
    min_per_class: int,
    seed: int,
) -> pd.DataFrame:
    """Per-class sample of `fraction` rows with a floor of `min_per_class`."""
    parts = []
    for _, group in data.groupby(label_column):
        n = min(len(group), max(round(len(group) * fraction), min_per_class))
        parts.append(group.sample(n=n, random_state=seed))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)


sample = stratified_sample(df, LABEL_COLUMN, SAMPLE_FRACTION, MIN_PER_CLASS, SEED)
print(f"Sample rows: {len(sample):,} ({len(sample) / len(df):.2%} of original)")

## Compare label distributions

Compare original and sample counts, shares, and the percentage retained for each label.

In [ ]:
sample_counts = sample[LABEL_COLUMN].value_counts().sort_index()
summary = pd.DataFrame(
    {
        "original": original_counts,
        "sample": sample_counts,
        "original_share_%": (original_counts / original_counts.sum() * 100).round(3),
        "sample_share_%": (sample_counts / sample_counts.sum() * 100).round(3),
        "kept_%": (sample_counts / original_counts * 100).round(2),
    }
)
summary.index = summary.index.map(label_names)
summary

## Save output

Save the sample under `datasets/`. Running this cell overwrites an existing sample.

In [ ]:
sample.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size / 1e6:.1f} MB)")